# ORCA PFZ ML - Notebook 02: PFZ Pseudo-Label Generation

**Purpose**: Generate scientifically justified PFZ pseudo-labels from real satellite data.

**Data Sources (verified accessible)**:
- NOAA OISST v2.1 — Sea Surface Temperature (0.25 deg daily)
- NASA MODIS-Aqua L3 — Chlorophyll-a (4km, 8-day composite)

**IMPORTANT DISCLAIMERS**:
- These are PSEUDO-LABELS derived from satellite-observed oceanographic conditions.
- They are NOT official INCOIS PFZ advisories.
- The labeling criteria replicate the scientific methodology used by INCOIS (thermal front + chlorophyll + optimal SST), but the labels themselves are generated from this pipeline, not obtained from INCOIS.
- This must be clearly documented in the model card.

**Scientific Basis**:
INCOIS generates PFZ advisories by identifying zones where:
1. Sharp SST gradients indicate thermal fronts (fish aggregate at boundaries)
2. Elevated chlorophyll-a indicates phytoplankton productivity (base of food chain)
3. SST is within optimal range for Indian pelagic fish species

We replicate this logic on the same satellite data to create training labels.

## Step 0: Install Dependencies and Setup

In [ ]:
!pip install xarray netCDF4 numpy pandas matplotlib scipy

In [ ]:
import xarray as xr
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.ndimage import sobel
from pathlib import Path
import os
import warnings
warnings.filterwarnings('ignore')

print(f"xarray: {xr.__version__}")
print(f"numpy: {np.__version__}")
print(f"pandas: {pd.__version__}")

In [ ]:
# Mount Google Drive (where raw data was saved in Notebook 01)
from google.colab import drive
drive.mount('/content/drive')

# Paths to raw data (saved by Notebook 01)
SST_DIR = Path('/content/drive/MyDrive/orca_ml/raw/sst')
CHL_DIR = Path('/content/drive/MyDrive/orca_ml/raw/chlorophyll')

# Output directory
OUTPUT_DIR = Path('/content/orca_data')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Indian Ocean bounding box
LAT_MIN, LAT_MAX = 7.0, 23.0
LON_MIN, LON_MAX = 66.0, 95.0

print(f"SST directory: {SST_DIR}")
print(f"Chlorophyll directory: {CHL_DIR}")
print(f"Output directory: {OUTPUT_DIR}")

## Step 1: Load SST and Chlorophyll Data

In [ ]:
# List available SST files
sst_files = sorted(SST_DIR.glob('oisst-avhrr-v02r01.*.nc'))
print(f"Available SST files: {len(sst_files)}")

if len(sst_files) == 0:
    raise FileNotFoundError(
        "No SST files found. Run Notebook 01 first to download NOAA OISST data."
    )

print(f"First: {sst_files[0].name}")
print(f"Last:  {sst_files[-1].name}")

In [ ]:
# List available Chlorophyll files
chl_files = sorted(CHL_DIR.glob('*.nc'))
print(f"Available Chlorophyll files: {len(chl_files)}")

if len(chl_files) == 0:
    print("WARNING: No chlorophyll files found.")
    print("If MODIS download hasn't completed, this notebook will use SST-only features.")
    print("For full PFZ labeling, chlorophyll is strongly recommended.")
    HAS_CHLOROPHYLL = False
else:
    print(f"First: {chl_files[0].name}")
    print(f"Last:  {chl_files[-1].name}")
    HAS_CHLOROPHYLL = True

In [ ]:
# Load a single SST file to inspect structure
ds_sst_sample = xr.open_dataset(sst_files[0])

# Subset to Indian Ocean
ds_sst_sample = ds_sst_sample.sel(lat=slice(LAT_MIN, LAT_MAX), lon=slice(LON_MIN, LON_MAX))

print("SST dataset structure (Indian Ocean subset):")
print(f"  Dimensions: {dict(ds_sst_sample.dims)}")
print(f"  Lat range: {float(ds_sst_sample.lat.min()):.2f} to {float(ds_sst_sample.lat.max()):.2f}")
print(f"  Lon range: {float(ds_sst_sample.lon.min()):.2f} to {float(ds_sst_sample.lon.max()):.2f}")
print(f"  Grid size: {len(ds_sst_sample.lat)} x {len(ds_sst_sample.lon)} = {len(ds_sst_sample.lat) * len(ds_sst_sample.lon)} cells")

# Extract SST values
sst_grid = ds_sst_sample['sst'].isel(time=0, zlev=0).values
valid_mask = ~np.isnan(sst_grid)
print(f"\n  Total grid cells: {sst_grid.size}")
print(f"  Ocean cells (valid): {valid_mask.sum()}")
print(f"  Land cells (NaN): {(~valid_mask).sum()}")

ds_sst_sample.close()

In [ ]:
# Load chlorophyll sample if available
if HAS_CHLOROPHYLL:
    ds_chl_sample = xr.open_dataset(chl_files[0])
    
    # MODIS lat may be in descending order
    if ds_chl_sample.lat[0] > ds_chl_sample.lat[-1]:
        ds_chl_sample = ds_chl_sample.sel(lat=slice(LAT_MAX, LAT_MIN), lon=slice(LON_MIN, LON_MAX))
    else:
        ds_chl_sample = ds_chl_sample.sel(lat=slice(LAT_MIN, LAT_MAX), lon=slice(LON_MIN, LON_MAX))
    
    print("Chlorophyll dataset structure (Indian Ocean subset):")
    print(f"  Dimensions: {dict(ds_chl_sample.dims)}")
    print(f"  Variables: {list(ds_chl_sample.data_vars)}")
    
    # Find chlorophyll variable name (may be 'chlor_a' or 'CHL' etc.)
    chl_var_name = None
    for var in ds_chl_sample.data_vars:
        if 'chlor' in var.lower() or 'chl' in var.lower():
            chl_var_name = var
            break
    
    if chl_var_name:
        print(f"  Chlorophyll variable: '{chl_var_name}'")
        chl_grid = ds_chl_sample[chl_var_name].values.squeeze()
        chl_valid = chl_grid[~np.isnan(chl_grid)]
        print(f"  Valid pixels: {len(chl_valid)}")
        print(f"  Range: {chl_valid.min():.4f} to {chl_valid.max():.4f} mg/m3")
    else:
        print(f"  WARNING: Could not identify chlorophyll variable in: {list(ds_chl_sample.data_vars)}")
        HAS_CHLOROPHYLL = False
    
    ds_chl_sample.close()

## Step 2: Calculate SST Spatial Gradient (Thermal Front Indicator)

Thermal fronts are detected by computing the spatial gradient magnitude of SST.
A high gradient indicates a sharp temperature change over a short distance — a frontal boundary where fish aggregate.

**Method**: Sobel filter applied to the 2D SST grid to compute gradient magnitude.

**Units**: The gradient is in deg C per grid cell. Since OISST grid spacing is 0.25 deg (~28km), a gradient of 0.5 means ~0.5 deg C change over 28km.

In [ ]:
def compute_sst_gradient(sst_2d):
    """
    Compute SST spatial gradient magnitude using Sobel filter.
    
    Parameters:
        sst_2d: 2D numpy array of SST values (may contain NaN for land)
    
    Returns:
        gradient_magnitude: 2D array of gradient magnitude (deg C per grid cell)
    """
    # Replace NaN with 0 for Sobel computation, track mask
    mask = np.isnan(sst_2d)
    sst_filled = np.where(mask, 0, sst_2d)
    
    # Sobel filter in lat (axis=0) and lon (axis=1) directions
    grad_lat = sobel(sst_filled, axis=0)
    grad_lon = sobel(sst_filled, axis=1)
    
    # Magnitude
    gradient_mag = np.sqrt(grad_lat**2 + grad_lon**2)
    
    # Restore NaN where original data was land
    gradient_mag[mask] = np.nan
    
    # Also NaN out pixels immediately adjacent to land (edge artifacts)
    from scipy.ndimage import binary_dilation
    land_buffer = binary_dilation(mask, iterations=1)
    gradient_mag[land_buffer & ~mask] = np.nan
    
    return gradient_mag

In [ ]:
# Compute gradient for sample file
ds_sst_sample = xr.open_dataset(sst_files[0])
ds_sst_sample = ds_sst_sample.sel(lat=slice(LAT_MIN, LAT_MAX), lon=slice(LON_MIN, LON_MAX))

sst_grid = ds_sst_sample['sst'].isel(time=0, zlev=0).values
gradient_grid = compute_sst_gradient(sst_grid)

# Statistics on gradient
grad_valid = gradient_grid[~np.isnan(gradient_grid)]

print("SST Spatial Gradient Statistics (sample day):")
print(f"  Valid pixels: {len(grad_valid)}")
print(f"  Min:    {grad_valid.min():.4f} deg C / grid cell")
print(f"  Max:    {grad_valid.max():.4f} deg C / grid cell")
print(f"  Mean:   {grad_valid.mean():.4f} deg C / grid cell")
print(f"  Median: {np.median(grad_valid):.4f} deg C / grid cell")
print(f"  Std:    {grad_valid.std():.4f} deg C / grid cell")
print(f"\nPercentiles:")
for pct in [50, 75, 90, 95, 99]:
    print(f"  P{pct}: {np.percentile(grad_valid, pct):.4f}")

ds_sst_sample.close()

In [ ]:
# Visualize SST gradient
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# SST
im1 = axes[0].imshow(sst_grid, origin='lower', cmap='RdYlBu_r', vmin=22, vmax=31,
                      extent=[LON_MIN, LON_MAX, LAT_MIN, LAT_MAX], aspect='auto')
axes[0].set_title('Sea Surface Temperature (deg C)', fontsize=12)
axes[0].set_xlabel('Longitude')
axes[0].set_ylabel('Latitude')
plt.colorbar(im1, ax=axes[0], label='SST (deg C)')

# Gradient
im2 = axes[1].imshow(gradient_grid, origin='lower', cmap='hot_r', vmin=0, vmax=np.percentile(grad_valid, 95),
                      extent=[LON_MIN, LON_MAX, LAT_MIN, LAT_MAX], aspect='auto')
axes[1].set_title('SST Gradient Magnitude (Thermal Front Indicator)', fontsize=12)
axes[1].set_xlabel('Longitude')
axes[1].set_ylabel('Latitude')
plt.colorbar(im2, ax=axes[1], label='Gradient (deg C / grid cell)')

plt.tight_layout()
plt.savefig(str(OUTPUT_DIR / 'sst_gradient_visualization.png'), dpi=150, bbox_inches='tight')
plt.show()

print("High-gradient areas (bright in right plot) indicate thermal fronts.")

## Step 3: Inspect SST Distribution

In [ ]:
# SST distribution for Indian Ocean
sst_valid = sst_grid[~np.isnan(sst_grid)]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Histogram
axes[0].hist(sst_valid, bins=60, color='steelblue', edgecolor='white', alpha=0.8)
axes[0].axvline(x=26.0, color='orange', linestyle='--', label='26 deg C (lower fish optimal)')
axes[0].axvline(x=30.0, color='red', linestyle='--', label='30 deg C (upper fish optimal)')
axes[0].set_xlabel('SST (deg C)')
axes[0].set_ylabel('Pixel Count')
axes[0].set_title('SST Distribution - Indian Ocean')
axes[0].legend(fontsize=9)

# Gradient histogram
axes[1].hist(grad_valid, bins=80, color='coral', edgecolor='white', alpha=0.8, range=(0, np.percentile(grad_valid, 99)))
axes[1].set_xlabel('SST Gradient (deg C / grid cell)')
axes[1].set_ylabel('Pixel Count')
axes[1].set_title('SST Gradient Distribution')

plt.tight_layout()
plt.savefig(str(OUTPUT_DIR / 'sst_distributions.png'), dpi=150, bbox_inches='tight')
plt.show()

print(f"\nSST Summary:")
print(f"  Mean: {sst_valid.mean():.2f} deg C")
print(f"  Std:  {sst_valid.std():.2f} deg C")
print(f"  Pixels in optimal range (26-30 C): {((sst_valid >= 26) & (sst_valid <= 30)).sum()} ({((sst_valid >= 26) & (sst_valid <= 30)).mean()*100:.1f}%)")

## Step 4: Analyze Chlorophyll Distribution

In [ ]:
if HAS_CHLOROPHYLL:
    # Load chlorophyll data
    ds_chl = xr.open_dataset(chl_files[0])
    
    if ds_chl.lat[0] > ds_chl.lat[-1]:
        ds_chl = ds_chl.sel(lat=slice(LAT_MAX, LAT_MIN), lon=slice(LON_MIN, LON_MAX))
    else:
        ds_chl = ds_chl.sel(lat=slice(LAT_MIN, LAT_MAX), lon=slice(LON_MIN, LON_MAX))
    
    chl_grid = ds_chl[chl_var_name].values.squeeze()
    chl_valid = chl_grid[~np.isnan(chl_grid)]
    
    # Chlorophyll is typically log-normal
    chl_log = np.log10(chl_valid[chl_valid > 0])
    
    fig, axes = plt.subplots(1, 3, figsize=(18, 5))
    
    # Raw distribution
    axes[0].hist(chl_valid, bins=100, color='green', edgecolor='white', alpha=0.8,
                 range=(0, np.percentile(chl_valid, 99)))
    axes[0].axvline(x=0.5, color='orange', linestyle='--', label='0.5 mg/m3')
    axes[0].axvline(x=1.0, color='red', linestyle='--', label='1.0 mg/m3')
    axes[0].set_xlabel('Chlorophyll-a (mg/m3)')
    axes[0].set_ylabel('Pixel Count')
    axes[0].set_title('Chlorophyll-a Distribution (raw)')
    axes[0].legend(fontsize=9)
    
    # Log distribution
    axes[1].hist(chl_log, bins=80, color='darkgreen', edgecolor='white', alpha=0.8)
    axes[1].axvline(x=np.log10(0.5), color='orange', linestyle='--', label='log10(0.5)')
    axes[1].axvline(x=np.log10(1.0), color='red', linestyle='--', label='log10(1.0)')
    axes[1].set_xlabel('log10(Chlorophyll-a)')
    axes[1].set_ylabel('Pixel Count')
    axes[1].set_title('Chlorophyll-a Distribution (log10)')
    axes[1].legend(fontsize=9)
    
    # Spatial map
    im = axes[2].imshow(np.log10(np.where(chl_grid > 0, chl_grid, np.nan)), origin='lower',
                        cmap='YlGn', extent=[LON_MIN, LON_MAX, LAT_MIN, LAT_MAX], aspect='auto')
    axes[2].set_title('log10(Chlorophyll-a) Spatial Map')
    axes[2].set_xlabel('Longitude')
    axes[2].set_ylabel('Latitude')
    plt.colorbar(im, ax=axes[2], label='log10(mg/m3)')
    
    plt.tight_layout()
    plt.savefig(str(OUTPUT_DIR / 'chlorophyll_distributions.png'), dpi=150, bbox_inches='tight')
    plt.show()
    
    print(f"\nChlorophyll-a Summary:")
    print(f"  Valid pixels: {len(chl_valid)}")
    print(f"  Mean:   {chl_valid.mean():.4f} mg/m3")
    print(f"  Median: {np.median(chl_valid):.4f} mg/m3")
    print(f"  Std:    {chl_valid.std():.4f} mg/m3")
    print(f"\n  Percentiles:")
    for pct in [25, 50, 75, 90, 95, 99]:
        print(f"    P{pct}: {np.percentile(chl_valid, pct):.4f} mg/m3")
    
    ds_chl.close()
else:
    print("Chlorophyll data not available. Will proceed with SST-only features.")
    print("NOTE: PFZ labels without chlorophyll are less reliable.")

## Step 5: Define PFZ Pseudo-Label Criteria

Based on the data distributions observed above and published scientific literature on Indian Ocean PFZ generation:

### Scientific Reasoning

1. **SST Gradient (Thermal Front)**: Fish aggregate at thermal boundaries. The gradient threshold should select the top ~10-20% of gradient values — these represent genuine frontal features, not background variability.

2. **Chlorophyll-a (Productivity)**: Elevated chlorophyll indicates phytoplankton blooms that attract forage fish, which in turn attract commercially important pelagic species. Coastal Indian waters typically show chlorophyll > 0.5 mg/m3 in productive zones.

3. **SST Range (Species Suitability)**: Indian pelagic fish species (sardines, mackerel, tuna) prefer waters in the 26-30 deg C range. Outside this range, fish aggregation is less likely.

### Threshold Determination
Thresholds are determined FROM THE DATA (percentile-based) rather than hardcoded arbitrarily. This ensures they adapt to the actual distribution of satellite observations.

In [ ]:
# Determine data-driven thresholds
# SST gradient: use P75 as threshold (top 25% indicates notable frontal activity)
GRADIENT_THRESHOLD = np.percentile(grad_valid, 75)

# SST optimal range for Indian pelagic fish
SST_MIN_OPTIMAL = 26.0  # deg C
SST_MAX_OPTIMAL = 30.0  # deg C

# Chlorophyll threshold
if HAS_CHLOROPHYLL:
    # Use P60 of valid chlorophyll — areas above this have above-average productivity
    CHL_THRESHOLD = np.percentile(chl_valid, 60)
    # Ensure it's at least 0.3 mg/m3 (minimum meaningful productivity)
    CHL_THRESHOLD = max(CHL_THRESHOLD, 0.3)
else:
    CHL_THRESHOLD = None

print("=" * 60)
print("PFZ PSEUDO-LABEL CRITERIA (data-driven thresholds)")
print("=" * 60)
print(f"\n1. SST Gradient Magnitude > {GRADIENT_THRESHOLD:.4f} deg C/grid cell")
print(f"   (P75 of observed gradients — top 25% = frontal zones)")
print(f"\n2. SST in optimal range: {SST_MIN_OPTIMAL} to {SST_MAX_OPTIMAL} deg C")
print(f"   (Published optimal range for Indian pelagic species)")
if CHL_THRESHOLD:
    print(f"\n3. Chlorophyll-a > {CHL_THRESHOLD:.4f} mg/m3")
    print(f"   (P60 of observed values — above-average productivity)")
else:
    print(f"\n3. Chlorophyll: NOT AVAILABLE (SST-only labeling)")
print(f"\nA grid cell is labeled PFZ=1 when ALL conditions are met simultaneously.")
print(f"\nDISCLAIMER: These are derived pseudo-labels, NOT official INCOIS PFZ advisories.")

## Step 6: Process All Files and Generate Training Dataset

For each SST file:
1. Load SST, subset to Indian Ocean
2. Compute SST gradient
3. Match with corresponding chlorophyll data (nearest 8-day composite)
4. Apply labeling criteria
5. Flatten grid to tabular format (one row per valid ocean pixel)

In [ ]:
def get_nearest_chlorophyll(target_date, chl_files):
    """
    Find the chlorophyll file whose date range contains or is nearest to target_date.
    MODIS files cover 8-day windows.
    """
    if not chl_files:
        return None
    
    # Extract dates from filenames (format: AQUA_MODIS.YYYYMMDD_YYYYMMDD...)
    best_file = None
    best_distance = float('inf')
    
    for f in chl_files:
        try:
            # Try to extract date from filename
            parts = f.stem.split('.')
            for part in parts:
                if '_' in part and len(part.split('_')[0]) == 8:
                    date_str = part.split('_')[0]
                    file_date = pd.Timestamp(date_str)
                    distance = abs((target_date - file_date).days)
                    if distance < best_distance:
                        best_distance = distance
                        best_file = f
                    break
        except (ValueError, IndexError):
            continue
    
    # Only use if within 8 days (one composite period)
    if best_distance <= 8:
        return best_file
    return None

In [ ]:
def process_single_day(sst_file, chl_files, chl_var_name):
    """
    Process one day of SST data into training samples.
    
    Returns a DataFrame with columns:
    date, latitude, longitude, sst, sst_gradient, chlorophyll, pfz_label
    """
    # Extract date from SST filename
    date_str = sst_file.stem.split('.')[-1]  # YYYYMMDD
    date = pd.Timestamp(date_str)
    
    # Load SST
    ds = xr.open_dataset(sst_file)
    ds = ds.sel(lat=slice(LAT_MIN, LAT_MAX), lon=slice(LON_MIN, LON_MAX))
    sst_grid = ds['sst'].isel(time=0, zlev=0).values
    lats = ds.lat.values
    lons = ds.lon.values
    ds.close()
    
    # Compute gradient
    gradient_grid = compute_sst_gradient(sst_grid)
    
    # Load chlorophyll if available
    chl_grid_resampled = None
    if HAS_CHLOROPHYLL and chl_files:
        chl_file = get_nearest_chlorophyll(date, chl_files)
        if chl_file is not None:
            try:
                ds_chl = xr.open_dataset(chl_file)
                if ds_chl.lat[0] > ds_chl.lat[-1]:
                    ds_chl = ds_chl.sel(lat=slice(LAT_MAX, LAT_MIN), lon=slice(LON_MIN, LON_MAX))
                else:
                    ds_chl = ds_chl.sel(lat=slice(LAT_MIN, LAT_MAX), lon=slice(LON_MIN, LON_MAX))
                
                # Regrid chlorophyll to SST grid using nearest-neighbor interpolation
                chl_data = ds_chl[chl_var_name].squeeze()
                chl_resampled = chl_data.interp(lat=lats, lon=lons, method='nearest')
                chl_grid_resampled = chl_resampled.values
                ds_chl.close()
            except Exception:
                chl_grid_resampled = None
    
    # Create meshgrid of coordinates
    lon_mesh, lat_mesh = np.meshgrid(lons, lats)
    
    # Valid ocean mask (not NaN in SST and not NaN in gradient)
    valid = ~np.isnan(sst_grid) & ~np.isnan(gradient_grid)
    
    # Flatten to 1D arrays
    flat_lat = lat_mesh[valid]
    flat_lon = lon_mesh[valid]
    flat_sst = sst_grid[valid]
    flat_grad = gradient_grid[valid]
    
    # Chlorophyll (may have additional NaN from cloud cover)
    if chl_grid_resampled is not None:
        flat_chl = chl_grid_resampled[valid]
    else:
        flat_chl = np.full(flat_sst.shape, np.nan)
    
    # Apply PFZ labeling criteria
    cond_gradient = flat_grad > GRADIENT_THRESHOLD
    cond_sst_range = (flat_sst >= SST_MIN_OPTIMAL) & (flat_sst <= SST_MAX_OPTIMAL)
    
    if CHL_THRESHOLD is not None and not np.all(np.isnan(flat_chl)):
        cond_chl = flat_chl > CHL_THRESHOLD
        # For pixels without chlorophyll data, cannot determine PFZ
        cond_chl_valid = ~np.isnan(flat_chl)
        pfz_label = (cond_gradient & cond_sst_range & cond_chl & cond_chl_valid).astype(int)
        # Exclude pixels where chlorophyll is NaN (can't make a determination)
        include_mask = cond_chl_valid
    else:
        # SST-only labeling (less reliable, document this)
        pfz_label = (cond_gradient & cond_sst_range).astype(int)
        include_mask = np.ones(len(flat_sst), dtype=bool)
    
    # Build DataFrame
    df = pd.DataFrame({
        'date': date,
        'latitude': flat_lat[include_mask],
        'longitude': flat_lon[include_mask],
        'sst': flat_sst[include_mask],
        'sst_gradient': flat_grad[include_mask],
        'chlorophyll': flat_chl[include_mask] if chl_grid_resampled is not None else np.nan,
        'pfz_label': pfz_label[include_mask]
    })
    
    return df

In [ ]:
# Process all available SST files
from tqdm import tqdm

all_dfs = []
errors = []

print(f"Processing {len(sst_files)} SST files...")
print(f"Chlorophyll available: {HAS_CHLOROPHYLL}")
print()

for sst_file in tqdm(sst_files, desc="Processing days"):
    try:
        df_day = process_single_day(sst_file, chl_files if HAS_CHLOROPHYLL else [], chl_var_name if HAS_CHLOROPHYLL else None)
        all_dfs.append(df_day)
    except Exception as e:
        errors.append((sst_file.name, str(e)))

print(f"\nProcessed: {len(all_dfs)} days successfully")
print(f"Errors: {len(errors)} days failed")

if errors:
    print("\nFirst 5 errors:")
    for fname, err in errors[:5]:
        print(f"  {fname}: {err}")

In [ ]:
# Combine all days into one dataset
df_full = pd.concat(all_dfs, ignore_index=True)

print("=" * 60)
print("COMBINED DATASET")
print("=" * 60)
print(f"\nTotal observations: {len(df_full):,}")
print(f"Date range: {df_full['date'].min()} to {df_full['date'].max()}")
print(f"Unique dates: {df_full['date'].nunique()}")
print(f"\nColumns: {list(df_full.columns)}")
print(f"\nMemory usage: {df_full.memory_usage(deep=True).sum() / 1024**2:.1f} MB")
print(f"\n{df_full.describe()}")

## Step 7: Label Assignment and Class Distribution

**Reminder**: pfz_label = 1 means ALL of these were true simultaneously:
- SST gradient > data-driven threshold (thermal front present)
- SST in 26-30 deg C range (species-suitable temperature)
- Chlorophyll > data-driven threshold (productive waters) [if available]

These are NOT official INCOIS labels.

In [ ]:
# Class distribution
class_counts = df_full['pfz_label'].value_counts().sort_index()
class_pcts = df_full['pfz_label'].value_counts(normalize=True).sort_index() * 100

print("=" * 60)
print("PFZ PSEUDO-LABEL CLASS DISTRIBUTION")
print("=" * 60)
print(f"\n  PFZ = 0 (Non-fishing zone):  {class_counts[0]:>10,} samples ({class_pcts[0]:.2f}%)")
print(f"  PFZ = 1 (Potential fishing):  {class_counts[1]:>10,} samples ({class_pcts[1]:.2f}%)")
print(f"  {'':->50}")
print(f"  Total:                        {len(df_full):>10,} samples")
print(f"\n  Class imbalance ratio: 1:{class_counts[0]//max(class_counts[1],1)}")
print(f"\n  NOTE: Class imbalance is expected. PFZ zones are a small")
print(f"  fraction of total ocean area on any given day.")
print(f"  The ML model will use class_weight='balanced' to handle this.")

In [ ]:
# Verify label quality — compare feature distributions between classes
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

pfz_0 = df_full[df_full['pfz_label'] == 0]
pfz_1 = df_full[df_full['pfz_label'] == 1]

# SST comparison
axes[0].hist(pfz_0['sst'].dropna(), bins=50, alpha=0.6, color='blue', label=f'Non-PFZ (n={len(pfz_0):,})', density=True)
axes[0].hist(pfz_1['sst'].dropna(), bins=50, alpha=0.6, color='red', label=f'PFZ (n={len(pfz_1):,})', density=True)
axes[0].set_xlabel('SST (deg C)')
axes[0].set_ylabel('Density')
axes[0].set_title('SST Distribution by Label')
axes[0].legend()

# Gradient comparison
axes[1].hist(pfz_0['sst_gradient'].dropna(), bins=50, alpha=0.6, color='blue', label='Non-PFZ', density=True,
             range=(0, np.percentile(df_full['sst_gradient'].dropna(), 99)))
axes[1].hist(pfz_1['sst_gradient'].dropna(), bins=50, alpha=0.6, color='red', label='PFZ', density=True,
             range=(0, np.percentile(df_full['sst_gradient'].dropna(), 99)))
axes[1].axvline(x=GRADIENT_THRESHOLD, color='black', linestyle='--', label=f'Threshold ({GRADIENT_THRESHOLD:.4f})')
axes[1].set_xlabel('SST Gradient')
axes[1].set_ylabel('Density')
axes[1].set_title('SST Gradient by Label')
axes[1].legend()

# Chlorophyll comparison
if HAS_CHLOROPHYLL and 'chlorophyll' in df_full.columns:
    chl_0 = pfz_0['chlorophyll'].dropna()
    chl_1 = pfz_1['chlorophyll'].dropna()
    if len(chl_0) > 0 and len(chl_1) > 0:
        upper = np.percentile(df_full['chlorophyll'].dropna(), 99)
        axes[2].hist(chl_0, bins=50, alpha=0.6, color='blue', label='Non-PFZ', density=True, range=(0, upper))
        axes[2].hist(chl_1, bins=50, alpha=0.6, color='red', label='PFZ', density=True, range=(0, upper))
        if CHL_THRESHOLD:
            axes[2].axvline(x=CHL_THRESHOLD, color='black', linestyle='--', label=f'Threshold ({CHL_THRESHOLD:.4f})')
        axes[2].set_xlabel('Chlorophyll-a (mg/m3)')
        axes[2].set_ylabel('Density')
        axes[2].set_title('Chlorophyll-a by Label')
        axes[2].legend()
    else:
        axes[2].text(0.5, 0.5, 'Insufficient chlorophyll data', ha='center', va='center', transform=axes[2].transAxes)
else:
    axes[2].text(0.5, 0.5, 'Chlorophyll not available', ha='center', va='center', transform=axes[2].transAxes)

plt.tight_layout()
plt.savefig(str(OUTPUT_DIR / 'label_feature_distributions.png'), dpi=150, bbox_inches='tight')
plt.show()

print("\nThe separation between classes confirms labels are meaningful:")
print("- PFZ pixels should have higher SST gradients (by definition)")
print("- PFZ pixels should have SST concentrated in 26-30 C range")
print("- PFZ pixels should have higher chlorophyll (if available)")

## Step 8: Spatial Visualization of PFZ Labels

In [ ]:
# Plot PFZ and non-PFZ locations on a map (using one sample day)
# Use the first day's data for visualization
sample_date = df_full['date'].iloc[0]
df_sample = df_full[df_full['date'] == sample_date].copy()

pfz_points = df_sample[df_sample['pfz_label'] == 1]
non_pfz_points = df_sample[df_sample['pfz_label'] == 0]

fig, ax = plt.subplots(1, 1, figsize=(14, 10))

# Plot non-PFZ as small blue dots
ax.scatter(non_pfz_points['longitude'], non_pfz_points['latitude'],
           s=1, c='lightsteelblue', alpha=0.3, label=f'Non-PFZ (n={len(non_pfz_points):,})')

# Plot PFZ as larger red dots
ax.scatter(pfz_points['longitude'], pfz_points['latitude'],
           s=8, c='red', alpha=0.7, label=f'PFZ (n={len(pfz_points):,})')

# Indian coastline reference points
coast_labels = [
    (80.27, 13.08, 'Chennai'),
    (76.27, 9.93, 'Kochi'),
    (70.38, 20.89, 'Veraval'),
    (83.35, 17.68, 'Vizag'),
    (88.36, 22.57, 'Kolkata'),
]
for lon, lat, name in coast_labels:
    ax.plot(lon, lat, 'k^', markersize=8)
    ax.annotate(name, (lon, lat), textcoords="offset points", xytext=(5, 5), fontsize=9)

ax.set_xlim(LON_MIN, LON_MAX)
ax.set_ylim(LAT_MIN, LAT_MAX)
ax.set_xlabel('Longitude (E)', fontsize=12)
ax.set_ylabel('Latitude (N)', fontsize=12)
ax.set_title(f'PFZ Pseudo-Labels — {sample_date.strftime("%Y-%m-%d")}\n'
             f'(Red = PFZ conditions met, Blue = Non-PFZ)', fontsize=13)
ax.legend(loc='upper left', fontsize=11)
ax.grid(True, alpha=0.3)
ax.set_aspect('equal')

plt.tight_layout()
plt.savefig(str(OUTPUT_DIR / 'pfz_spatial_map.png'), dpi=150, bbox_inches='tight')
plt.show()

print(f"\nSample day ({sample_date.strftime('%Y-%m-%d')}):")
print(f"  Total ocean pixels: {len(df_sample):,}")
print(f"  PFZ pixels: {len(pfz_points):,} ({len(pfz_points)/len(df_sample)*100:.2f}%)")
print(f"  Non-PFZ pixels: {len(non_pfz_points):,} ({len(non_pfz_points)/len(df_sample)*100:.2f}%)")

In [ ]:
# Additional: PFZ occurrence frequency map (across all days)
if df_full['date'].nunique() > 1:
    # For each grid cell, count how often it was labeled PFZ
    pfz_frequency = df_full.groupby(['latitude', 'longitude'])['pfz_label'].mean().reset_index()
    pfz_frequency.columns = ['latitude', 'longitude', 'pfz_frequency']
    
    # Only show cells that were PFZ at least once
    pfz_ever = pfz_frequency[pfz_frequency['pfz_frequency'] > 0]
    
    fig, ax = plt.subplots(1, 1, figsize=(14, 10))
    
    scatter = ax.scatter(pfz_ever['longitude'], pfz_ever['latitude'],
                         c=pfz_ever['pfz_frequency'], cmap='YlOrRd',
                         s=5, alpha=0.7, vmin=0, vmax=1)
    plt.colorbar(scatter, ax=ax, label='Fraction of days labeled PFZ')
    
    ax.set_xlim(LON_MIN, LON_MAX)
    ax.set_ylim(LAT_MIN, LAT_MAX)
    ax.set_xlabel('Longitude (E)')
    ax.set_ylabel('Latitude (N)')
    ax.set_title(f'PFZ Occurrence Frequency ({df_full["date"].nunique()} days)\n'
                 f'(Darker = more frequently labeled as PFZ)')
    ax.grid(True, alpha=0.3)
    ax.set_aspect('equal')
    
    plt.tight_layout()
    plt.savefig(str(OUTPUT_DIR / 'pfz_frequency_map.png'), dpi=150, bbox_inches='tight')
    plt.show()
    
    print(f"Grid cells that were PFZ at least once: {len(pfz_ever):,}")
    print(f"Grid cells never labeled PFZ: {len(pfz_frequency) - len(pfz_ever):,}")
else:
    print("Only one day of data — frequency map requires multiple days.")

## Step 9: Save Final Labeled Dataset

In [ ]:
# Final cleanup before saving
# Remove any rows where essential columns are NaN
essential_cols = ['date', 'latitude', 'longitude', 'sst', 'sst_gradient', 'pfz_label']
df_clean = df_full.dropna(subset=essential_cols).copy()

# Ensure correct dtypes
df_clean['pfz_label'] = df_clean['pfz_label'].astype(int)
df_clean['date'] = pd.to_datetime(df_clean['date'])

print(f"Rows before cleanup: {len(df_full):,}")
print(f"Rows after cleanup:  {len(df_clean):,}")
print(f"Rows removed:        {len(df_full) - len(df_clean):,}")

In [ ]:
# Save to CSV
output_path = OUTPUT_DIR / 'pfz_training_dataset.csv'
df_clean.to_csv(output_path, index=False)

file_size_mb = output_path.stat().st_size / (1024 * 1024)

print(f"\nDataset saved to: {output_path}")
print(f"File size: {file_size_mb:.2f} MB")

In [ ]:
# Also save to Google Drive for persistence
drive_output = Path('/content/drive/MyDrive/orca_ml/processed')
drive_output.mkdir(parents=True, exist_ok=True)
drive_path = drive_output / 'pfz_training_dataset.csv'
df_clean.to_csv(drive_path, index=False)
print(f"Also saved to Google Drive: {drive_path}")

## Step 10: Final Summary Report

In [ ]:
print("=" * 70)
print("ORCA PFZ ML — LABELED DATASET SUMMARY")
print("=" * 70)

print(f"\n{'Dataset Shape:':<30} {df_clean.shape[0]:,} rows x {df_clean.shape[1]} columns")
print(f"{'Columns:':<30} {list(df_clean.columns)}")
print(f"{'Date range:':<30} {df_clean['date'].min().strftime('%Y-%m-%d')} to {df_clean['date'].max().strftime('%Y-%m-%d')}")
print(f"{'Unique dates:':<30} {df_clean['date'].nunique()}")
print(f"{'Lat range:':<30} {df_clean['latitude'].min():.2f} to {df_clean['latitude'].max():.2f}")
print(f"{'Lon range:':<30} {df_clean['longitude'].min():.2f} to {df_clean['longitude'].max():.2f}")

print(f"\n{'--- Class Distribution ---':^70}")
class_counts = df_clean['pfz_label'].value_counts().sort_index()
class_pcts = df_clean['pfz_label'].value_counts(normalize=True).sort_index() * 100
print(f"{'  PFZ = 0 (Non-PFZ):':<30} {class_counts[0]:>10,} ({class_pcts[0]:.2f}%)")
print(f"{'  PFZ = 1 (PFZ):':<30} {class_counts[1]:>10,} ({class_pcts[1]:.2f}%)")
print(f"{'  Imbalance ratio:':<30} 1:{class_counts[0]//max(class_counts[1],1)}")

print(f"\n{'--- Feature Statistics ---':^70}")
print(df_clean[['sst', 'sst_gradient', 'chlorophyll']].describe().round(4).to_string())

print(f"\n{'--- Labeling Criteria Used ---':^70}")
print(f"  SST gradient > {GRADIENT_THRESHOLD:.4f} (P75 of data)")
print(f"  SST in [{SST_MIN_OPTIMAL}, {SST_MAX_OPTIMAL}] deg C")
if CHL_THRESHOLD:
    print(f"  Chlorophyll > {CHL_THRESHOLD:.4f} mg/m3 (P60 of data)")
else:
    print(f"  Chlorophyll: NOT USED (data unavailable)")

print(f"\n{'--- Data Sources ---':^70}")
print(f"  SST: NOAA OISST v2.1 (ncei.noaa.gov)")
print(f"  Chlorophyll: NASA MODIS-Aqua L3 (oceandata.sci.gsfc.nasa.gov)")

print(f"\n{'--- Output ---':^70}")
print(f"  File: {output_path}")
print(f"  Size: {file_size_mb:.2f} MB")

print(f"\n{'--- DISCLAIMER ---':^70}")
print(f"  These are PSEUDO-LABELS derived from satellite observations.")
print(f"  They are NOT official INCOIS PFZ advisories.")
print(f"  The labeling criteria replicate INCOIS methodology but")
print(f"  the labels are generated by this pipeline.")

print(f"\n{'--- NEXT STEP ---':^70}")
print(f"  Proceed to Notebook 03 for ML model training.")
print(f"  DO NOT modify this dataset after training begins.")
print("\n" + "=" * 70)